# Reproduce the dementia results (Papers 2, 7, 8)
A CPU runtime is enough. Runtime → Run all. Downloads OpenNeuro ds004504 (participants table + denoised derivatives).

In [ ]:
!git clone -q https://github.com/aftab802/eeg-dementia-integrity-benchmark
%cd eeg-dementia-integrity-benchmark
!pip -q install mne pymatreader xgboost imbalanced-learn eeglabio edfio awscli

In [ ]:
!aws s3 sync --no-sign-request --only-show-errors s3://openneuro.org/ds004504 data/ds004504 --exclude '*' --include 'participants.tsv' --include 'dataset_description.json' --include 'derivatives/*'
!ls data/ds004504/derivatives | head -3; ls data/ds004504/derivatives | wc -l

In [ ]:
# Paper 7 / Paper 2 Study A: label correction, 39 features, six classifiers, tuned XGBoost + nested CV
!python scripts/run_label_correction.py --bids-root data/ds004504 2>&1 | grep -v ': done' | tail -20

In [ ]:
# Original notebook protocol (epoch features, subject-grouped CV, subject voting)
!python scripts/run_subject_vote_xgb.py --bids-root data/ds004504 2>&1 | tail -4

In [ ]:
# LEAD-EEG pipeline on the public AD-vs-HC subset (proxy, not the 176-subject cohort)
!python scripts/run_lead_eeg.py --ds004504 data/ds004504 2>&1 | tail -4

In [ ]:
import glob, json
out = {}
for f in sorted(glob.glob('results/*.json')):
    r = json.load(open(f)); k = f.split('/')[-1]
    if 'six_classifier_benchmark' in r:
        out[k] = {'n': r['n_subjects'], 'features': r['n_features'], 'bench': r['six_classifier_benchmark'], 'xgb': r['xgb_gridsearch'],
                  'top15_outside_cv': r['top15_selection_outside_cv'], 'top15_inside_cv': r['top15_selection_inside_cv'],
                  'misassigned_positional': r['rows_misassigned_by_positional_join']}
    elif 'groupkfold' in r:
        out[k] = {'groupkfold_subject_acc': r['groupkfold']['subject_accuracy'], 'holdout_acc': r['holdout_accuracy'], 'classes': r['classes']}
    else:
        out[k] = {kk: r.get(kk) for kk in ('cohort', 'n_subjects', 'accuracy_pooled', 'accuracy_fold_mean', 'accuracy_fold_sd', 'confusion_matrix')}
        out[k]['dementia'] = r['report'].get('1'); out[k]['healthy'] = r['report'].get('0')
print('RESULTS_JSON_BEGIN'); print(json.dumps(out)); print('RESULTS_JSON_END')